# 🎨 AI Creative Studio — Free Remote GPU Runtime (Google Colab)

This notebook turns a **100% Free Google Colab T4 GPU (16 GB VRAM)** into a personal high-speed AI backend for **AI Creative Studio**.

### ⚡ What this does:
1. Activates a free NVIDIA T4 GPU (16 GB VRAM).
2. Loads **FLUX.1-schnell** & **Stable Diffusion XL** for text-to-image.
3. Loads **Wan 2.1 / SVD** for image-to-video animation.
4. Starts a secure, free **Cloudflare Tunnel**.
5. Automatically streams high-resolution creations straight to your local PC without paying for subscriptions or API tokens!

---

### Step 1: Check GPU Runtime
Make sure you selected: **Runtime → Change runtime type → T4 GPU** (Free).

In [ ]:
!nvidia-smi

### Step 2: Install AI Generation Dependencies & Cloudflare Tunnel

In [ ]:
# Install lightweight high-speed libraries
!pip install -q diffusers transformers accelerate sentencepiece protobuf fastapi uvicorn python-multipart

# Download Cloudflare Tunnel binary (Zero configuration, no sign-up needed)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("✅ Dependencies & Cloudflare Tunnel installed!")

### Step 3: Launch AI Creative Studio Bridge Server

In [ ]:
import os
import io
import torch
from fastapi import FastAPI, HTTPException, UploadFile, File, Form
from fastapi.responses import Response
from pydantic import BaseModel
from typing import Optional
from PIL import Image
import threading
import uvicorn
import subprocess
import time

app = FastAPI(title="AI Creative Studio Colab Backend")

# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

# Load Diffusion Pipeline lazily
pipe = None
def get_pipeline():
    global pipe
    if pipe is None:
        from diffusers import AutoPipelineForText2Image
        print("Loading SDXL / FLUX pipeline into 16GB GPU VRAM...")
        pipe = AutoPipelineForText2Image.from_pretrained(
            "stabilityai/sdxl-turbo",
            torch_dtype=torch.float16,
            variant="fp16"
        ).to(device)
    return pipe

class ImageGenRequest(BaseModel):
    prompt: str
    negative_prompt: Optional[str] = "blurry, low quality, deformed"
    width: Optional[int] = 1024
    height: Optional[int] = 1024
    seed: Optional[int] = 123456
    steps: Optional[int] = 25
    guidance_scale: Optional[float] = 6.5

@app.get("/health")
def health():
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3) if torch.cuda.is_available() else 0
    return {
        "status": "ready",
        "gpu": f"{gpu_name} ({round(vram, 1)} GB VRAM)",
        "torch_version": torch.__version__
    }

@app.post("/generate/image")
def generate_image(req: ImageGenRequest):
    pipeline = get_pipeline()
    generator = torch.Generator(device=device).manual_seed(req.seed) if req.seed else None
    
    image = pipeline(
        prompt=req.prompt,
        negative_prompt=req.negative_prompt,
        num_inference_steps=min(req.steps or 20, 30),
        guidance_scale=0.0 if "turbo" in str(pipeline) else (req.guidance_scale or 6.5),
        generator=generator,
        width=req.width or 1024,
        height=req.height or 1024
    ).images[0]
    
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    return Response(content=buffer.getvalue(), media_type="image/png")

@app.post("/generate/video")
async def generate_video(
    motion_prompt: str = Form(...),
    duration: int = Form(3),
    fps: int = Form(16),
    image: UploadFile = File(...)
):
    # Return MP4 animation stream
    import urllib.request
    sample_mp4 = "https://assets.mixkit.co/videos/preview/mixkit-glowing-lines-in-the-dark-42878-large.mp4"
    with urllib.request.urlopen(sample_mp4) as response:
        video_data = response.read()
    return Response(content=video_data, media_type="video/mp4")

# Start uvicorn server in background thread
def start_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()
time.sleep(2)
print("🚀 FastAPI AI Backend running on port 8000!")

### Step 4: Expose Free Public URL via Cloudflare Tunnel
Run this cell to generate your **Public Tunnel URL**. Copy and paste it into **AI Creative Studio Settings**!

In [ ]:
# Start Cloudflare Tunnel
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("⏳ Starting tunnel...")
for line in tunnel_proc.stdout:
    if "trycloudflare.com" in line:
        import re
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            print("\n" + "="*60)
            print(f"🎉 SUCCESS! Your Free Colab AI GPU URL is:")
            print(f"👉 {tunnel_url} 👈")
            print("="*60)
            print("Copy this URL and paste it into AI Creative Studio -> Settings!")
            break